### Computing attention weight of a single input token


In [2]:
import torch

#already tokenized and embedded 
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)
#select word "journey" to calculate attention scores related to it
query = inputs[1]



#### 1. compute raw attention scores based on dot product 

In [3]:
attn_scores_2=torch.empty(inputs.shape[0]) # empty tensor to store attention scores (would be 6 since 6 tokens/words)
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i,query)

#unnomalized attention scores for "journey" (x^2) with respect to all tokens/words in the input
print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


#### 2. compute raw attention scores based on dot product 


In [4]:
#apply softmax to get attention weights (normalized)
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

attn_weights_2_naive = softmax_naive(attn_scores_2)
print("Attention weights:", attn_weights_2_naive)
print("Sum:", attn_weights_2_naive.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


In [5]:
#using Pytorch's built-in softmax function for comparison (better performance)
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print("Attention weights (PyTorch):", attn_weights_2)
print("Sum (PyTorch):", attn_weights_2.sum())

Attention weights (PyTorch): tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum (PyTorch): tensor(1.)


#### 3. Forming Context Vectors

In [6]:
context_vec_2 = torch.zeros(query.shape) #vectors with all zeros same size as embedding dimensions
for i, x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i] * x_i #weighted sum 
print("Context vector for 'journey':", context_vec_2)

Context vector for 'journey': tensor([0.4419, 0.6515, 0.5683])


### Computing attention weight of all tokens 


In [10]:
#compute attention scores for all pairs based on dot product
attn_scores = torch.empty(6,6)
for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_scores[i][j] = torch.dot(x_i,x_j)
print("Attention scores matrix:\n", attn_scores)

#for loops are inefficient
attn_scores = inputs @ inputs.T
print("Attention scores matrix (optimized):\n", attn_scores)

Attention scores matrix:
 tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])
Attention scores matrix (optimized):
 tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [13]:
#atten weights normailized across rows (for each query)
attn_weights = torch.softmax(attn_scores, dim = -1) #-1-> last dimension and calculated softmax across the columns for every row (every row is adds to 1)
print("Attention weights matrix:\n", attn_weights)

row_2_sum = sum([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
print("Row 2 sum:", row_2_sum)
print("All rows sun: ", attn_weights.sum(dim=-1))

Attention weights matrix:
 tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])
Row 2 sum: 1.0
All rows sun:  tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


In [14]:
all_context_vecs = attn_weights @ inputs
print("Context vectors for all tokens:\n", all_context_vecs)

Context vectors for all tokens:
 tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])
